# Weak OT and Barycentric Projection

This notebook generates `fig:weak-ot-barycentric-projection`.  For a coupling
$$
    \pi(dx,dy)=\pi_x(dy)\alpha(dx),
$$
weak barycentric costs keep only the conditional mean
$$
    \bar y(x)=\int y\,d\pi_x(y).
$$
The figure shows both the full conditional spread and the shorter source-to-barycenter displacement seen by the weak cost.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

NAME = "weak-ot-barycentric-projection"
out = figure_dir(NAME)
rng = np.random.default_rng(284)

The coupling is feasible by construction: each source atom splits its mass among nearby target atoms, and target masses are the column sums of the displayed plan.

In [2]:
x = np.array([
    [-1.05,  0.74],
    [-1.20,  0.30],
    [-1.04, -0.08],
    [-1.24, -0.50],
    [-0.88, -0.76],
    [-0.78,  0.02],
])
centers = np.array([[0.70, 0.70], [1.15, 0.00], [0.76, -0.72]])
y = np.vstack([
    rng.normal(loc=centers[0], scale=[0.10, 0.11], size=(5, 2)),
    rng.normal(loc=centers[1], scale=[0.11, 0.10], size=(5, 2)),
    rng.normal(loc=centers[2], scale=[0.10, 0.11], size=(5, 2)),
])
preferred = np.array([
    [0.62, 0.58],
    [0.93, 0.34],
    [1.04, 0.04],
    [0.97, -0.28],
    [0.68, -0.62],
    [0.96, 0.04],
])
P = np.zeros((len(x), len(y)))
for i, p in enumerate(preferred):
    d2 = np.sum((y - p)**2, axis=1)
    idx = np.argsort(d2)[:5]
    w = np.exp(-d2[idx] / 0.075)
    w = w / w.sum()
    P[i, idx] = w / len(x)
a = P.sum(axis=1)
b = P.sum(axis=0)
bar = (P @ y) / a[:, None]
pairs = [(i, j, float(P[i, j])) for i in range(P.shape[0]) for j in range(P.shape[1]) if P[i, j] > 1e-10]
all_points = np.vstack([x, y, bar])
xlim, ylim = padded_limits(all_points, pad=0.16)

In [3]:
def decorate(ax):
    ax.set_aspect("equal")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    remove_axes(ax)

fig, ax = plt.subplots(figsize=(2.45, 2.05))
draw_transport_segments(ax, x, y, pairs, color=VIOLET, min_width=0.15, max_width=1.35, alpha_scale=0.50, zorder=1)
draw_point_clouds(ax, x, y, source_weights=a, target_weights=b, base_size=DIRAC_MARKER_SIZE * 0.84)
decorate(ax)
save_pdf(fig, out / "conditional-coupling.pdf", pad_inches=0.055)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.45, 2.05))
draw_transport_segments(ax, x, y, pairs, color=LIGHT_GRAY, min_width=0.12, max_width=0.90, alpha_scale=0.20, zorder=1)
for i in range(len(x)):
    ax.plot([x[i, 0], bar[i, 0]], [x[i, 1], bar[i, 1]], color=VIOLET, lw=1.05, alpha=0.86, zorder=2)
ax.scatter(bar[:, 0], bar[:, 1], s=DIRAC_MARKER_SIZE * 0.78, marker="o", color=VIOLET, edgecolor="none", linewidth=0, zorder=4)
draw_point_clouds(ax, x, y, source_weights=a, target_weights=b, base_size=DIRAC_MARKER_SIZE * 0.84)
decorate(ax)
save_pdf(fig, out / "barycentric-projection.pdf", pad_inches=0.055)
plt.close(fig)